<a href="https://colab.research.google.com/github/eunyeongkimm/multimodal_user_needs_understanding/blob/main/results/09_audio_native_model_test/qwen3_omni_test_vllm_prompt.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# ===== 셀 1 : vLLM + PyTorch 호환 버전 설치 =====

!pip uninstall -y -q vllm vllm-omni torch torchvision torchaudio

# PyTorch / TorchAudio를 완전히 같은 버전 + CUDA 13.0으로 설치
!pip install -q --no-cache-dir \
    torch==2.11.0 \
    torchvision==0.26.0 \
    torchaudio==2.11.0 \
    --index-url https://download.pytorch.org/whl/cu130

# vLLM 및 Qwen 오디오 의존성 설치
!pip install -q --no-cache-dir \
    vllm \
    qwen-omni-utils \
    soundfile \
    librosa

print("✅ 설치 완료")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 366.1/366.1 MB 136.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 169.9/169.9 MB 133.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 196.5/196.5 MB 114.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.4/60.4 MB 140.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 MB 121.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 304.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 214.1/214.1 MB 123.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 341.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 143.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.5/59.5 MB 122.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 200.9/200.9 MB 140.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 145.9/145.9 MB 120.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━

In [2]:
# CUDA 13 라이브러리 경로 등록
!echo "/usr/local/lib/python3.12/dist-packages/nvidia/cu13/lib" \
    | sudo tee /etc/ld.so.conf.d/nvidia-cu13.conf

!sudo ldconfig

# 제대로 잡혔는지 확인
!ldconfig -p | grep libcudart

/usr/local/lib/python3.12/dist-packages/nvidia/cu13/lib
/sbin/ldconfig.real: /usr/local/lib/libtbb.so.12 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libumf.so.1 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libtbbbind_2_5.so.3 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libur_adapter_level_zero.so.0 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libtbbbind_2_0.so.3 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libtbbmalloc_proxy.so.2 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libur_adapter_level_zero_v2.so.0 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libtbbbind.so.3 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libur_loader.so.0 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libhwloc.so.15 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libtcm_debug.so.1 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libtcm.so.1 is not a symbolic

In [1]:
import torch
import torchaudio
import vllm

print("torch:", torch.__version__)
print("CUDA:", torch.version.cuda)
print("torchaudio:", torchaudio.__version__)
print("vllm:", vllm.__version__)

print("GPU:", torch.cuda.get_device_name(0))

torch: 2.13.0+cu130
CUDA: 13.0
torchaudio: 2.11.0+cu130
vllm: 0.27.1
GPU: NVIDIA A100-SXM4-40GB


In [2]:
# ===== 셀 2 : 구글드라이브 마운트 =====
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd, os
AUDIO_ROOT = '/content/drive/MyDrive/audio_seg'
PARQUET_DIR = '/content/drive/MyDrive/audio_seg_2'

manifest = pd.read_parquet(os.path.join(PARQUET_DIR, 'audio_seg_manifest.parquet'))
gold = pd.read_parquet(os.path.join(PARQUET_DIR, 'gold_actual_batch1_final.parquet'))
gold_map = dict(zip(gold.call_id, gold.label))
print("manifest:", manifest.shape, "| gold:", gold.shape)

Mounted at /content/drive
manifest: (1119, 11) | gold: (19847, 5)


In [3]:
import torch
print("free:", round(torch.cuda.mem_get_info()[0]/1e9,1), "GB / total:", round(torch.cuda.mem_get_info()[1]/1e9,1), "GB")

free: 42.0 GB / total: 42.4 GB


In [4]:
# ===== vLLM 로드: Colab 안전 버전 =====

import os
import sys

os.environ["VLLM_ENABLE_V1_MULTIPROCESSING"] = "0"

# DEBUG 쓰지 않음
os.environ["VLLM_LOGGING_LEVEL"] = "WARNING"
os.environ["TRANSFORMERS_VERBOSITY"] = "error"

# progress bar도 최대한 억제
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"

MODEL_ID = "cyankiwi/Qwen3-Omni-30B-A3B-Thinking-AWQ-4bit"

# 원래 Colab 출력 보관
_original_stdout = sys.stdout
_original_stderr = sys.stderr

# 실제 파일 객체 → fileno() 지원
_vllm_log = open("/tmp/vllm_load.log", "w")

# ★ vLLM import 전에 변경
sys.stdout = _vllm_log
sys.stderr = _vllm_log

load_error = None

try:
    from vllm import LLM, SamplingParams

    llm = LLM(
        model=MODEL_ID,
        runner="generate",
        trust_remote_code=True,
        tensor_parallel_size=1,

        gpu_memory_utilization=0.90,
        max_model_len=4096,
        max_num_seqs=1,

        limit_mm_per_prompt={
            "audio": 5,
            "image": 1,
            "video": 0,
        },

        enforce_eager=True,
        generation_config="vllm",
    )

except Exception as e:
    load_error = e

finally:
    # Colab 화면 출력 복구
    sys.stdout = _original_stdout
    sys.stderr = _original_stderr

    # 일부 logger가 이 파일을 계속 참조할 수 있으므로
    # 여기서는 일부러 close하지 않음
    _vllm_log.flush()


if load_error is None:
    print("✅ vLLM loaded OK")

else:
    print("❌ vLLM load failed:")
    print(repr(load_error))

    # 전체 로그 절대 출력하지 않고 마지막 40줄만
    with open("/tmp/vllm_load.log", "r", errors="replace") as f:
        lines = f.readlines()

    print("\n===== 마지막 로그 40줄 =====")
    print("".join(lines[-40:]))

INFO 08-15 22:51:57 [api_utils.py:273] non-default args: {'runner': 'generate', 'trust_remote_code': True, 'max_model_len': 4096, 'gpu_memory_utilization': 0.9, 'max_num_seqs': 1, 'disable_log_stats': True, 'enforce_eager': True, 'limit_mm_per_prompt': {'audio': 5, 'image': 1, 'video': 0}, 'generation_config': 'vllm', 'model': 'cyankiwi/Qwen3-Omni-30B-A3B-Thinking-AWQ-4bit'}
WARNING 08-15 22:51:58 [arg_utils.py:1678] The global random seed is set to 0. Since VLLM_ENABLE_V1_MULTIPROCESSING is set to False, this may affect the random state of the Python process that launched vLLM.


INFO 08-15 22:52:12 [model.py:645] Resolved architecture: Qwen3OmniMoeForConditionalGeneration
INFO 08-15 22:52:12 [model.py:1883] Using max model len 4096
INFO 08-15 22:52:18 [scheduler.py:242] Chunked prefill is enabled with max_num_batched_tokens=4096.
WARNING 08-15 22:52:18 [vllm.py:1194] Enforce eager set, disabling torch.compile and CUDAGraphs. This is equivalent to setting -cc.mode=none -cc.cudagraph_mode=none
WARNING 08-15 22:52:18 [vllm.py:1247] Inductor compilation was disabled by user settings, optimizations settings that are only active during inductor compilation will be ignored.
INFO 08-15 22:52:18 [kernel.py:306] Final IR op priority after setting platform defaults: IrOpPriorityConfig(rms_norm=['vllm_c', 'native'], fused_add_rms_norm=['vllm_c', 'native'])
INFO 08-15 22:52:18 [vllm.py:1426] Cudagraph is disabled under eager mode
INFO 08-15 22:52:18 [compilation.py:329] Enabled custom fusions: norm_quant, act_quant
INFO 08-15 22:52:36 [core.py:121] Initializing a V1 LLM en

Loading safetensors checkpoint shards:   0% Completed | 0/5 [00:00<?, ?it/s]


INFO 08-15 22:54:07 [default_loader.py:430] Loading weights took 33.31 seconds
INFO 08-15 22:54:07 [int_wna16.py:409] Using MoEPrepareAndFinalizeNoDPEPModular
INFO 08-15 22:54:07 [int_wna16.py:410] Using MarlinExperts
INFO 08-15 22:54:10 [gpu_model_runner.py:5405] Model loading took 19.22 GiB memory and 90.392979 seconds
INFO 08-15 22:54:11 [gpu_model_runner.py:6465] Encoder cache will be initialized with a budget of 12544 tokens, and profiled with 1 image items of the maximum feature size.
INFO 08-15 22:55:57 [gpu_worker.py:563] Available KV cache memory: 14.11 GiB
INFO 08-15 22:55:57 [kv_cache_utils.py:2235] GPU KV cache size: 154,160 tokens
INFO 08-15 22:55:57 [kv_cache_utils.py:2236] Maximum concurrency for 4,096 tokens per request: 37.64x
INFO 08-15 22:55:57 [gpu_worker.py:789] Free memory on device (39.08/39.49 GiB) on startup. Desired GPU memory utilization is (0.9, 35.54 GiB). Actual usage is 19.91 GiB for consumed memory (weights + non-torch), 1.52 GiB for peak activation, and

In [5]:
# ===== 헬퍼 함수 복구 셀 =====

import os
import re

CATEGORIES = [
    "환불요청",
    "주문취소",
    "불만제기",
    "배송확인",
    "교환반품",
    "구매진행",
    "서비스이용",
]


def get_call_wavs(call_id, max_utterances=5):
    rows = (
        manifest[manifest.call_id == call_id]
        .sort_values("utt_idx")
        .head(max_utterances)
    )

    paths = []

    for _, r in rows.iterrows():
        p = os.path.join(
            AUDIO_ROOT,
            call_id,
            os.path.basename(r.wav_path)
        )

        if os.path.exists(p):
            paths.append(p)

    return paths


def build_prompt(n_audio):
    audio_tags = "".join(
        f"발화{i}: "
        "<|audio_start|><|audio_pad|><|audio_end|>\n"
        for i in range(1, n_audio + 1)
    )

    cats = ", ".join(CATEGORIES)

    return (
        "<|im_start|>system\n"
        "당신은 한국어 콜센터 통화 분류기입니다. "
        "고객의 초반 발화 음성을 듣고 의도를 분류합니다."
        "<|im_end|>\n"

        "<|im_start|>user\n"
        f"{audio_tags}\n"

        f"위 발화들을 종합해 고객 의도를 "
        f"다음 7개 중 하나로 분류하세요: {cats}\n"

        "최종 답변은 반드시 "
        "'정답: <카테고리>' 형식으로 작성하세요."
        "<|im_end|>\n"

        "<|im_start|>assistant\n"
    )


def parse_pred(resp):
    # "정답:" 뒤에 나오는 카테고리만 인정 (엄격)
    m = re.search(r"정답\s*[:：]\s*\n?\s*([가-힣]+)", resp)
    if m and m.group(1).strip() in CATEGORIES:
        return m.group(1).strip()
    # "정답:"이 아예 없고 응답이 안 잘렸으면(짧게 끝) 마지막 카테고리 허용
    # 단 "정답:"이 있는데 뒤가 빈 경우(잘림)는 None 처리
    if "정답" in resp:
        return None  # 정답 태그는 있는데 카테고리 못 뽑음 = 잘림/실패
    # 정답 태그 자체가 없으면 tail fallback
    tail = resp[-80:]
    for c in CATEGORIES:
        if c in tail:
            return c
    return None


print("✅ helper functions loaded")

✅ helper functions loaded


### 기본뼈대

In [7]:
# ===== 셀: 카테고리 정의 + 추론 허용=====
import time, librosa, re
from vllm import SamplingParams

CATEGORIES = ["환불요청","주문취소","불만제기","배송확인","교환반품","구매진행","서비스이용"]

# 카테고리 정의
CATEGORY_DEF = """
[카테고리 정의]
1. 환불요청: 결제한 금액을 돌려받는 것이 최종 목적인 경우로 보이면
   - "취소", "반품", "반송"이라는 단어가 나와도, 최종 목적이 금전 반환(현금/캐시)으로
     보이면 환불요청으로 분류할 것
2. 주문취소: 배송/수강 전, 환불 절차 없이 순수 주문 취소로 보이는 경우
3. 불만제기: 문의처럼 들리나 본질은 항의(약속 불이행, 응대 불만, 반복 통화에 대한
   불만 등)로 보이는 경우
4. 배송확인: 배송 상태·도착 문의로 보이는 경우
5. 교환반품: 불량·오배송으로 물건을 다른 물건으로 교체(금전 반환 아님)하려는
   것으로 보이는 경우
6. 구매진행: 결제 완료를 위한 도움을 요청하는 것으로 보이는 경우
7. 서비스이용: 로그인·기기·앱·시스템 이용 관련 문제로 보이는 경우
"""


# 추론 허용 — structured 안 씀, max_tokens 넉넉히
sampling = SamplingParams(temperature=0.0, max_tokens=3072)

def build_prompt(n_audio):
    audio_tags = "".join(
        f"발화{i}: <|audio_start|><|audio_pad|><|audio_end|>\n" for i in range(1, n_audio+1)
    )
    cats = ", ".join(CATEGORIES)
    return (
        "<|im_start|>system\n"
        "당신은 한국어 콜센터 고객 의도 분류기입니다. 고객의 초반 발화 음성을 듣고 전체 통화 의도를 분류합니다.\n\n"
        f"[카테고리 정의]\n{CATEGORY_DEF}\n\n"
        f"가능한 카테고리: {cats}\n"
        "<|im_end|>\n"
        "<|im_start|>user\n"
        f"{audio_tags}\n"
        "위 발화들을 듣고 고객 의도를 분류하세요.\n"
        "한국어로, 핵심 근거만 2문장 이내로 짧게 분석한 뒤 정답을 쓰세요.\n"
        "형식:\n"
        "분석: (2문장 이내)\n"
        "정답: (위 7개 중 하나)\n"
        "<|im_end|>\n"
        "<|im_start|>assistant\n"
    )

def parse_pred(resp):
    # "정답:" 우선, 없으면 응답 끝쪽 카테고리
    m = re.search(r"정답\s*[:：]\s*([가-힣]+)", resp)
    if m and m.group(1).strip() in CATEGORIES:
        return m.group(1).strip()
    tail = resp[-150:]
    for c in CATEGORIES:
        if c in tail:
            return c
    for c in CATEGORIES:
        if c in resp:
            return c
    return None

def predict_call(call_id):
    wavs = get_call_wavs(call_id, max_utterances=5)
    if not wavs:
        return {"call_id": call_id, "pred": None, "raw": "no_audio", "n_audio": 0}
    audios = [(librosa.load(w, sr=16000, mono=True)[0], 16000) for w in wavs]
    out = llm.generate(
        [{"prompt": build_prompt(len(audios)), "multi_modal_data": {"audio": audios}}],
        sampling_params=sampling, use_tqdm=False,
    )
    r = out[0].outputs[0]
    return {"call_id": call_id, "pred": parse_pred(r.text), "raw": r.text,
            "finish": r.finish_reason, "n_tok": len(r.token_ids), "n_audio": len(audios)}

# ---- 5콜 테스트 (추론 붙는지 + 잘림 없는지 확인) ----
gold_map = dict(zip(gold.call_id, gold.label))
for cid in manifest.call_id.unique()[:5]:
    t0 = time.time()
    r = predict_call(cid)
    print(f"\n=== {cid}  gold={gold_map.get(cid)}  pred={r['pred']}  "
          f"({time.time()-t0:.1f}s, {r['n_tok']}tok, {r.get('finish')}) ===")
    print(r["raw"][:700])


=== J16_S000434  gold=서비스이용  pred=None  (182.8s, 2048tok, length) ===
<think>
Okay, let's tackle this problem. So, the user provided five speech segments in Korean, and I need to classify the customer's intent into one of the seven categories. Let me go through each utterance carefully.

First, the user says "여보세요. 아까는 뭐냐?" which translates to "Hello. What was that earlier?" Then the next part is "엠베스 쓰여가고 엘리아이 보이." Hmm, maybe "엠베스" is a typo or mispronunciation. Wait, maybe it's "엠베스" as in "Mebes" but that doesn't make sense. Alternatively, maybe it's "엠베스" as a brand or product name. Then "엘리아이 보이" might be "Eliai boi" but that's unclear. Maybe "엘리아이" is "Eliai" but perhaps it's a mispronunciation of "엘리트" or something else. Wait, maybe "엠베스" is "Mebes" but

=== J16_S000633  gold=서비스이용  pred=서비스이용  (54.6s, 605tok, stop) ===
<think>
Okay, let's tackle this problem. So, the user provided five speech segments in Korean, and I need to classify the customer's intent into one of the seve

### think-off

In [9]:
# ===== think-OFF: 정의 + 즉답 (빠름) =====
import time, librosa, re, pandas as pd
from vllm import SamplingParams

sampling_off = SamplingParams(temperature=0.0, max_tokens=512)

def build_prompt_off(n_audio):
    audio_tags = "".join(f"발화{i}: <|audio_start|><|audio_pad|><|audio_end|>\n" for i in range(1, n_audio+1))
    cats = ", ".join(CATEGORIES)
    return (
        "<|im_start|>system\n"
        "당신은 한국어 콜센터 고객 의도 분류기입니다. 고객의 초반 발화 음성을 듣고 전체 통화 의도를 분류합니다.\n\n"
        f"[카테고리 정의]\n{CATEGORY_DEF}\n\n"
        f"가능한 카테고리: {cats}\n<|im_end|>\n"
        "<|im_start|>user\n"
        f"{audio_tags}\n"
        "위 발화들을 듣고 고객 의도를 분류하세요.\n형식:\n분석: (1문장)\n정답: (위 7개 중 하나)\n<|im_end|>\n"
        "<|im_start|>assistant\n<think>\n\n</think>\n\n"
    )

def predict_off(call_id):
    wavs = get_call_wavs(call_id, max_utterances=5)
    if not wavs:
        return {"call_id": call_id, "pred": None, "raw": "no_audio", "finish": None, "n_tok": 0}
    audios = [(librosa.load(w, sr=16000, mono=True)[0], 16000) for w in wavs]
    out = llm.generate([{"prompt": build_prompt_off(len(audios)), "multi_modal_data": {"audio": audios}}],
                       sampling_params=sampling_off, use_tqdm=False)
    r = out[0].outputs[0]
    return {"call_id": call_id, "pred": parse_pred(r.text), "raw": r.text, "finish": r.finish_reason, "n_tok": len(r.token_ids)}

# 5콜 확인
for cid in manifest.call_id.unique()[:5]:
    r = predict_off(cid)
    print(f"{cid} gold={gold_map.get(cid)} pred={r['pred']} finish={r['finish']} tok={r['n_tok']}")
    print(r["raw"][:150], "\n")

J16_S000434 gold=서비스이용 pred=서비스이용 finish=stop tok=568
<think>
Okay, let's tackle this problem. So, the user provided five different utterances in Korean, and I need to classify the customer's intent into  

J16_S000633 gold=서비스이용 pred=서비스이용 finish=stop tok=88
분석: 고객은 과거에 기기 변경을 여러 번 했고, 이번에 다시 기기 변경을 시도했으나 로그인 문제가 발생해 진행이 어려워졌다고 설명하고 있습니다. 이는 로그인 및 기기 변경 과정에서 발생한 시스템 문제로 보이며, 서비스 이용 관련 문의로 판단됩니다.
정답: 서비스이용 

J16_S000687 gold=서비스이용 pred=서비스이용 finish=stop tok=78
분석: 고객이 "관리자 인증"과 "인증번호가 일치하지 않습니다"라는 문장을 반복적으로 말하며, 로그인 또는 시스템 접근 관련 문제가 발생한 것으로 보입니다. 이는 서비스 이용 중 발생한 문제로, "서비스이용" 카테고리에 해당합니다.
정답: 서비스이용 

J16_S000727 gold=서비스이용 pred=서비스이용 finish=stop tok=51
분석: 고객의 발화는 "네 알겠습니다 감사합니다"로, 단순히 확인 및 감사 표현만 포함되어 있으며, 구체적인 의도나 요청 사항이 없음.
정답: 서비스이용 

J16_S000746 gold=서비스이용 pred=서비스이용 finish=stop tok=57
분석: 고객은 인터넷 강의 시청 중 서버 오류로 강의 재생이 되지 않는 문제를 설명하며, 서비스 이용 시 발생한 기술적 문제를 해결해 달라고 요청하고 있습니다.
정답: 서비스이용 



In [10]:
# think-off 250
rows = []
t0 = time.time()
for i, cid in enumerate(manifest.call_id.unique()):
    r = predict_off(cid); r["gold"] = gold_map.get(cid); rows.append(r)
    if (i+1) % 25 == 0: print(f"{i+1}/250 ({time.time()-t0:.0f}s)")
df_off = pd.DataFrame(rows)
df_off.to_parquet("/content/drive/MyDrive/audio_seg_2/qwen3_30b_def_thinkoff.parquet")
p = df_off.dropna(subset=["pred"])
print(f"\nacc: {(p.pred==p.gold).mean():.3f}, 파싱실패: {df_off.pred.isna().sum()}, length: {(df_off.finish=='length').sum()}")
print("예측:\n", df_off.pred.value_counts())

25/250 (231s)
50/250 (543s)
75/250 (830s)
100/250 (1110s)
125/250 (1385s)
150/250 (1652s)
175/250 (1919s)
200/250 (2236s)
225/250 (2559s)
250/250 (2852s)

acc: 0.546, 파싱실패: 1, length: 3
예측:
 pred
환불요청     69
배송확인     52
서비스이용    43
교환반품     28
불만제기     23
주문취소     19
구매진행     15
Name: count, dtype: int64


In [12]:
from sklearn.metrics import f1_score, precision_score, recall_score, confusion_matrix

d = df_off.dropna(subset=["pred"]).copy()
CATS = ["환불요청","주문취소","불만제기","배송확인","교환반품","구매진행","서비스이용"]

# macro-F1 + 불만 지표
macro = f1_score(d.gold, d.pred, labels=CATS, average="macro", zero_division=0)
f1_b = f1_score(d.gold, d.pred, labels=["불만제기"], average="micro", zero_division=0)
p_b = precision_score(d.gold, d.pred, labels=["불만제기"], average="micro", zero_division=0)
r_b = recall_score(d.gold, d.pred, labels=["불만제기"], average="micro", zero_division=0)
print(f"macro-F1: {macro:.3f}")
print(f"불만 F1: {f1_b:.3f} / P: {p_b:.3f} / R: {r_b:.3f}")

# 불만→환불 셀 (그 18셀)
cm = confusion_matrix(d.gold, d.pred, labels=CATS)
gi, pi = CATS.index("불만제기"), CATS.index("환불요청")
print(f"\n불만→환불 셀: {cm[gi][pi]}  (GPT-B는 19)")

# gold=불만제기 행 전체
print("\ngold=불만제기 → 예측 분포:")
for j, c in enumerate(CATS):
    if cm[gi][j] > 0: print(f"  {c}: {cm[gi][j]}")

macro-F1: 0.450
불만 F1: 0.356 / P: 0.565 / R: 0.260

불만→환불 셀: 16  (GPT-B는 19)

gold=불만제기 → 예측 분포:
  환불요청: 16
  주문취소: 1
  불만제기: 13
  배송확인: 12
  교환반품: 6
  구매진행: 1
  서비스이용: 1


In [13]:
for _, r in df_off.head(10).iterrows():
    print(r.pred, "|", r.raw[:120])

구매진행 | 분석: 고객은 "엠베스티"와 "엘리아이"라는 단어를 언급하며, "이거 이재용 봐요"라고 말해 제품 관련 문의를 하고 있는 것으로 보입니다. 이는 구매 진행 과정에서의 문의로 판단됩니다.
정답: 구매진행
서비스이용 | 분석: 고객은 과거에 기기 변경을 여러 번 했으며, 이번에 다시 기기 변경을 시도했으나 로그인 문제가 발생해 진행이 어려워졌다고 설명하고 있습니다. 이는 기기 변경 과정에서 발생한 로그인 문제로 인한 서비스 이용 장
서비스이용 | 분석: 고객이 "어제처럼 눌러요"라고 말하며, 관리자 인증이 필요하다는 점을 언급하고, 인증번호가 일치하지 않는다는 내용을 전달하며, 전용탭 폐쇄 및 초기화 관련 문제를 제기하고 있습니다. 이는 시스템 사용 중 발생
서비스이용 | 분석: 고객의 발화는 "네 알겠습니다 감사합니다"로, 명확한 의도를 나타내지 않으며 단순히 감사 표현만 포함되어 있습니다. 이는 통화의 종료를 알리는 말로 보이나, 의도 분류를 위해 추가 정보가 필요합니다.
정답: 
서비스이용 | 분석: 고객은 인터넷 강의 시청 중 서버 오류로 강의 재생이 되지 않는 문제를 설명하며, 서비스 이용 시 발생한 기술적 문제를 해결해 달라고 요청하고 있습니다.
정답: 서비스이용
서비스이용 | 분석: 고객은 서비스 등록 기간 정보가 5일 이상 변경되었다고 문의하며, 등록한 지 2일밖에 안 됐음에도 불구하고 문제가 발생했다고 주장하고 있습니다. 이는 서비스 이용 관련 문제로 보이며, 등록 정보 변경에 대한 
서비스이용 | 분석: 고객이 "지금 괜찮은가요"라고 물어보며, 이전 발화에서 "디비" 관련 문제와 "정지가 됐다"는 내용이 있었으나, 현재 상태 확인을 위해 문의하는 것으로 보임.
정답: 서비스이용
불만제기 | 분석: 고객은 "20분 전에 그 계획이 중복됐다고 막 빨리 끊어서 삭제해달라고 잘못 드렸거든요. 그래서 삭제해준다고 했는데 삭제 안 돼 가지고...". 이는 계획 중복으로 인해 삭제 요청이 이루어졌으나 처리되지 않아
서비스이용 | 분석: 고객은 기

In [16]:
d = df_off.dropna(subset=["pred"])
from sklearn.metrics import f1_score
print("파싱성공만 macro-F1:", round(f1_score(d.gold, d.pred, labels=CATS, average="macro", zero_division=0), 3))
print("n:", len(d))

파싱성공만 macro-F1: 0.45
n: 249


### think - on

In [17]:
# ===== think-ON: 청크 저장 =====
import time, librosa, pandas as pd, os
from vllm import SamplingParams

sampling_on = SamplingParams(temperature=0.0, max_tokens=3072)
CKPT = "/content/drive/MyDrive/audio_seg_2/qwen3_30b_def_thinkon.parquet"

# build_prompt는 think 켜진 버전 (assistant 줄이 "<|im_start|>assistant\n"으로 끝, <think></think> 안 닫음)
def build_prompt_on(n_audio):
    audio_tags = "".join(f"발화{i}: <|audio_start|><|audio_pad|><|audio_end|>\n" for i in range(1, n_audio+1))
    cats = ", ".join(CATEGORIES)
    return (
        "<|im_start|>system\n"
        "당신은 한국어 콜센터 고객 의도 분류기입니다. 고객의 초반 발화 음성을 듣고 전체 통화 의도를 분류합니다.\n\n"
        f"[카테고리 정의]\n{CATEGORY_DEF}\n\n"
        f"가능한 카테고리: {cats}\n<|im_end|>\n"
        "<|im_start|>user\n"
        f"{audio_tags}\n"
        "위 발화들을 듣고 고객 의도를 분류하세요.\n핵심 근거를 간단히 분석한 뒤 정답을 쓰세요.\n형식:\n분석: (간단히)\n정답: (위 7개 중 하나)\n<|im_end|>\n"
        "<|im_start|>assistant\n"
    )

def predict_on(call_id):
    wavs = get_call_wavs(call_id, max_utterances=5)
    if not wavs:
        return {"call_id": call_id, "pred": None, "raw": "no_audio", "finish": None, "n_tok": 0}
    audios = [(librosa.load(w, sr=16000, mono=True)[0], 16000) for w in wavs]
    out = llm.generate([{"prompt": build_prompt_on(len(audios)), "multi_modal_data": {"audio": audios}}],
                       sampling_params=sampling_on, use_tqdm=False)
    r = out[0].outputs[0]
    return {"call_id": call_id, "pred": parse_pred(r.text), "raw": r.text[-400:], "finish": r.finish_reason, "n_tok": len(r.token_ids)}

# 이미 한 것 로드 (재개)
done = set()
rows = []
if os.path.exists(CKPT):
    prev = pd.read_parquet(CKPT); rows = prev.to_dict("records"); done = set(prev.call_id)
    print(f"재개: {len(done)}개 완료됨")

t0 = time.time()
all_ids = list(manifest.call_id.unique())
for i, cid in enumerate(all_ids):
    if cid in done: continue
    r = predict_on(cid); r["gold"] = gold_map.get(cid); rows.append(r)
    if len(rows) % 25 == 0:   # 25개마다 저장
        pd.DataFrame(rows).to_parquet(CKPT)
        print(f"{len(rows)}/250 저장 ({time.time()-t0:.0f}s)")
pd.DataFrame(rows).to_parquet(CKPT)
print("완료 저장:", len(rows))

100/250 저장 (12172s)
125/250 저장 (14861s)
150/250 저장 (17635s)
175/250 저장 (20861s)
200/250 저장 (24243s)
225/250 저장 (27540s)
250/250 저장 (30707s)
완료 저장: 250


In [6]:
import pandas as pd
from sklearn.metrics import f1_score, precision_score, recall_score, confusion_matrix

CKPT = "/content/drive/MyDrive/audio_seg_2/qwen3_30b_def_thinkon.parquet"
CATEGORIES = ["환불요청","주문취소","불만제기","배송확인","교환반품","구매진행","서비스이용"]

df = pd.read_parquet(CKPT)
print(f"저장된 콜: {len(df)}/250")

# gold 없으면 붙이기 (혹시 저장 시 누락 대비)
if "gold" not in df.columns or df.gold.isna().any():
    gold = pd.read_parquet("/content/drive/MyDrive/audio_seg_2/gold_actual_batch1_final.parquet")
    gmap = dict(zip(gold.call_id, gold.label))
    df["gold"] = df.call_id.map(gmap)

# 잘림/파싱 현황
print("\nfinish:", df.finish.value_counts().to_dict())
print(f"length(잘림): {(df.finish=='length').sum()}/{len(df)} = {(df.finish=='length').mean()*100:.1f}%")
print(f"pred=None(파싱실패): {df.pred.isna().sum()}")

# 성능 (파싱 성공만)
d = df.dropna(subset=["pred"])
macro = round(f1_score(d.gold, d.pred, labels=CATEGORIES, average="macro", zero_division=0), 3)
f1_b = round(f1_score(d.gold, d.pred, labels=["불만제기"], average="micro", zero_division=0), 3)
p_b = round(precision_score(d.gold, d.pred, labels=["불만제기"], average="micro", zero_division=0), 3)
r_b = round(recall_score(d.gold, d.pred, labels=["불만제기"], average="micro", zero_division=0), 3)
acc = round((d.pred==d.gold).mean(), 3)

print(f"\n=== think-on 성능 (n={len(d)}) ===")
print(f"macro-F1: {macro}")
print(f"불만 F1: {f1_b} / P: {p_b} / R: {r_b}")
print(f"accuracy: {acc}")

# 18셀
cm = confusion_matrix(d.gold, d.pred, labels=CATEGORIES)
gi, pi = CATEGORIES.index("불만제기"), CATEGORIES.index("환불요청")
print(f"\n불만→환불 셀: {cm[gi][pi]}  (GPT-B 19, audio-only 16)")

print("\ngold=불만제기 → 예측 분포:")
for j, c in enumerate(CATEGORIES):
    if cm[gi][j] > 0: print(f"  {c}: {cm[gi][j]}")

print("\n예측 분포:\n", df.pred.value_counts())
print("\ngold 분포:\n", df.gold.value_counts())

저장된 콜: 250/250

finish: {'stop': 225, 'length': 25}
length(잘림): 25/250 = 10.0%
pred=None(파싱실패): 18

=== think-on 성능 (n=232) ===
macro-F1: 0.461
불만 F1: 0.378 / P: 0.519 / R: 0.298
accuracy: 0.591

불만→환불 셀: 21  (GPT-B 19, audio-only 16)

gold=불만제기 → 예측 분포:
  환불요청: 21
  불만제기: 14
  배송확인: 3
  교환반품: 7
  구매진행: 2

예측 분포:
 pred
환불요청     91
서비스이용    34
교환반품     28
배송확인     28
불만제기     27
구매진행     14
주문취소     10
Name: count, dtype: int64

gold 분포:
 gold
환불요청     87
서비스이용    51
불만제기     50
배송확인     24
교환반품     19
구매진행     11
주문취소      8
Name: count, dtype: int64


### think-off + 전사텍스트

In [9]:
# ===== 전사+원음: 정의 + 전사 텍스트 + 원음 (think-off) =====
import time, librosa, re, pandas as pd
from vllm import SamplingParams

sampling_tx = SamplingParams(temperature=0.0, max_tokens=512)

def get_call_utts(call_id):
    # (wav_path, text) 순서대로 최대 5개
    rows = manifest[manifest.call_id == call_id].sort_values("utt_idx")
    utts = []
    for _, r in rows.iterrows():
        p = os.path.join(AUDIO_ROOT, call_id, os.path.basename(r.wav_path))
        if os.path.exists(p):
            utts.append((p, str(r.text)))
    return utts[:5]

def build_prompt_tx(utts):
    # 발화마다 오디오 + 전사 나란히
    blocks = ""
    for i, (_, txt) in enumerate(utts, 1):
        blocks += f"발화{i}: <|audio_start|><|audio_pad|><|audio_end|> 전사: \"{txt}\"\n"
    cats = ", ".join(CATEGORIES)
    return (
        "<|im_start|>system\n"
        "당신은 한국어 콜센터 고객 의도 분류기입니다. 각 발화의 음성(원음)과 전사 텍스트를 함께 참고해 전체 통화 의도를 분류합니다.\n\n"
        f"[카테고리 정의]\n{CATEGORY_DEF}\n\n"
        f"가능한 카테고리: {cats}\n<|im_end|>\n"
        "<|im_start|>user\n"
        f"{blocks}\n"
        "위 발화들의 내용(전사)과 말투(음성)를 종합해 고객 의도를 분류하세요.\n"
        "형식:\n분석: (1문장)\n정답: (위 7개 중 하나)\n<|im_end|>\n"
        "<|im_start|>assistant\n<think>\n\n</think>\n\n"
    )

def predict_tx(call_id):
    utts = get_call_utts(call_id)
    if not utts:
        return {"call_id": call_id, "pred": None, "raw": "no_audio", "finish": None, "n_tok": 0}
    audios = [(librosa.load(p, sr=16000, mono=True)[0], 16000) for p, _ in utts]
    out = llm.generate([{"prompt": build_prompt_tx(utts), "multi_modal_data": {"audio": audios}}],
                       sampling_params=sampling_tx, use_tqdm=False)
    r = out[0].outputs[0]
    return {"call_id": call_id, "pred": parse_pred(r.text), "raw": r.text, "finish": r.finish_reason, "n_tok": len(r.token_ids)}

# 5콜 먼저
for cid in manifest.call_id.unique()[:5]:
    r = predict_tx(cid)
    print(f"{cid} gold={gold_map.get(cid)} pred={r['pred']} finish={r['finish']}")
    print(r["raw"][:180], "\n")

WARNING 08-15 23:03:07 [jit_monitor.py:135] Triton kernel JIT compilation during inference: _triton_mrope_forward. This causes a latency spike; consider extending warmup to cover this shape/config.
J16_S000434 gold=서비스이용 pred=배송확인 finish=stop
분석: 고객은 "엠자로 써 있어요?"라고 물어보며, "엠베스트하고 엘리하이 보이거."라고 언급하며 제품 또는 서비스 관련 정보를 확인하거나 문의하는 것으로 보입니다. 이는 배송 상태나 제품 정보 확인과 관련된 문의로, 배송확인 카테고리에 해당합니다.
정답: 배송확인 

J16_S000633 gold=서비스이용 pred=서비스이용 finish=stop
분석: 고객은 작년부터 기기 변경을 시도했으나 올해 재수를 하면서 패스를 구매한 후 기기 변경이 누적되어 실패했으며, 로그인이 막혀 문제가 발생했다고 설명하며, 이는 시스템 이용 관련 문제로 보인다.
정답: 서비스이용 

J16_S000687 gold=서비스이용 pred=서비스이용 finish=stop
분석: 고객은 관리자 인증 과정에서 인증번호 불일치로 인해 문제를 겪고 있으며, 전용탭 해제 및 초기화 옵션을 확인하며 시스템 사용 관련 문제 해결을 요청하는 것으로 보입니다.
정답: 서비스이용 

J16_S000727 gold=서비스이용 pred=서비스이용 finish=stop
분석: 고객이 "아, 네 알겠습니다. 감사합니다."라고 말하며, 음성에서 감사의 뉘앙스와 끝맺음이 강조되어 있으나, 구체적인 요청이나 불만 제기 등은 없어 단순히 대화를 마무리하는 것으로 보임.
정답: 서비스이용 

J16_S000746 gold=서비스이용 pred=서비스이용 finish=stop
분석: 고객은 인터넷 강의 시청 중 서버 오류로 인해 강의 재생이 불가능하다고 반복적으로 설명하며, 이는 시스템 이용 

In [10]:
# 전사+원음 250
rows = []
t0 = time.time()
for i, cid in enumerate(manifest.call_id.unique()):
    r = predict_tx(cid); r["gold"] = gold_map.get(cid); rows.append(r)
    if (i+1) % 25 == 0: print(f"{i+1}/250 ({time.time()-t0:.0f}s)")
df_tx = pd.DataFrame(rows)
df_tx.to_parquet("/content/drive/MyDrive/audio_seg_2/qwen3_30b_transcript.parquet")

from sklearn.metrics import f1_score, precision_score, recall_score, confusion_matrix
d = df_tx.dropna(subset=["pred"])
print(f"\nmacro-F1: {round(f1_score(d.gold,d.pred,labels=CATEGORIES,average='macro',zero_division=0),3)}")
print(f"불만 F1: {round(f1_score(d.gold,d.pred,labels=['불만제기'],average='micro',zero_division=0),3)}")
print(f"불만 R: {round(recall_score(d.gold,d.pred,labels=['불만제기'],average='micro',zero_division=0),3)}")
cm = confusion_matrix(d.gold,d.pred,labels=CATEGORIES)
gi,pi = CATEGORIES.index("불만제기"), CATEGORIES.index("환불요청")
print(f"불만→환불 셀: {cm[gi][pi]} (GPT-B 19, audio-only 16)")
print(f"acc: {round((d.pred==d.gold).mean(),3)}, 파싱실패: {df_tx.pred.isna().sum()}")

25/250 (223s)
50/250 (456s)
75/250 (683s)
100/250 (921s)
125/250 (1132s)
150/250 (1361s)
175/250 (1586s)
200/250 (1818s)
225/250 (2058s)
250/250 (2285s)

macro-F1: 0.482
불만 F1: 0.444
불만 R: 0.4
불만→환불 셀: 16 (GPT-B 19, audio-only 16)
acc: 0.572, 파싱실패: 0
